<a href="https://colab.research.google.com/github/ryouy/GraduationThesis/blob/main/1015/white_bear_and_zhou_pilot_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# White Bear / Negation — Pilot v3
## Reproducible Colab experiments: Mann-style behavioral tests, Zhou-style negation, and concept probes

**Target:** `Qwen/Qwen2.5-0.5B-Instruct`, free Google Colab T4 GPU. **No API key is needed for the model.**

**Research integrity:** The direction of the results is **not predetermined**. All conditions and examples are fixed before analysis. This notebook is a **small, exploratory analogue**, **not** a faithful replication of the published models, datasets or causal interventions.

**Relevant publications**
- [Mann et al. (2025), Don't Think of the White Bear](https://arxiv.org/abs/2511.12381)
- [Zhou et al. (2026), How Language Models Process Negation](https://proceedings.mlr.press/v306/zhou26bq.html); [official replication code](https://github.com/Ja1Zhou/LM_Negation)

### Changes from v2
1. Mann: target **singular and plural** counting; deduplicated zero-distractor baseline; common random seeds across conditions; generation text kept for manual inspection.
2. Zhou: report **word-answer likelihoods** as the primary behavior measure; preserve A/B/C as a diagnostic and add **contextual calibration** to examine label/position bias. Include the label selection matrix.
3. Internal representations: retain label Logit Lens and add **regularized linear concept probes** trained on separate examples. Check performance on held-out lexical templates **and** descriptions without the target word. Do not interpret weak probes as evidence of concept activation.
4. Save **raw CSV, plots, metadata, summary, ZIP**; optional secure GitHub upload from Colab Secrets (disabled by default).


## 0 — Runtime and fixed configuration

In Colab: **Runtime → Change runtime type → T4 GPU**, then **Run all**. The last GitHub upload cell is enabled; set the Colab secret `GITHUB_RESULTS_TOKEN` before running it. For a quick run, reduce `N_GENERATIONS`; for a more reliable experiment, increase it and later expand the number of distinct concepts.

In [ ]:
%pip -q install "transformers>=4.46,<5" "accelerate>=1,<2" "huggingface-hub>=0.30,<1" scikit-learn pandas numpy matplotlib tqdm requests

In [ ]:
import os, re, json, math, zipfile, random, hashlib, sys, platform
from pathlib import Path
from datetime import datetime, timezone
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = torch.float16 if DEVICE == 'cuda' else torch.float32
N_GENERATIONS = 5                # independent stochastic draws per unique prompt
DISTRACTOR_LEVELS = [0, 1, 3]   # zero is run only once, without fake related/unrelated split
PROBE_LAYERS = [8, 16, 24]      # hidden state 0=embedding, 24=final for this Qwen model
OUTPUT_DIR = Path('white_bear_zhou_v3_results')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=False)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=DTYPE, low_cpu_mem_usage=True, trust_remote_code=False).to(DEVICE).eval()
assert max(PROBE_LAYERS) <= model.config.num_hidden_layers
print('Model:', MODEL_ID, '| device:', DEVICE, '| transformer layers:', model.config.num_hidden_layers)

def chat(user_text):
    return tok.apply_chat_template([{'role':'user','content':user_text}], tokenize=False, add_generation_prompt=True)

def write_plot(name):
    plt.tight_layout(); plt.savefig(OUTPUT_DIR/name, dpi=180, bbox_inches='tight'); plt.show(); plt.close()

@torch.inference_mode()
def generate_one(user_text, seed, max_new_tokens=42):
    torch.manual_seed(int(seed))
    x=tok(chat(user_text), return_tensors='pt', add_special_tokens=False).to(DEVICE)
    y=model.generate(**x, max_new_tokens=max_new_tokens, do_sample=True, temperature=0.8, top_p=0.95,
                     pad_token_id=tok.eos_token_id)
    return tok.decode(y[0, x['input_ids'].shape[1]:], skip_special_tokens=True).strip()

@torch.inference_mode()
def continuation_logprob(user_text, completion):
    # Score exactly the text completed after the assistant header. A tokenization boundary
    # mismatch fails loudly rather than silently using incorrect token positions.
    prefix=chat(user_text)
    a=tok(prefix, add_special_tokens=False).input_ids
    b=tok(prefix+completion, add_special_tokens=False).input_ids
    if not b[:len(a)] == a or len(b) == len(a):
        raise ValueError(f'Token boundary mismatch for completion {completion!r}')
    z=torch.tensor([b], device=DEVICE)
    logp=model(z).logits[0].float().log_softmax(dim=-1)
    return float(sum(logp[k-1,b[k]].item() for k in range(len(a),len(b)))), len(b)-len(a)


## 1 — Stimuli: fixed in advance

Each target has a paired replacement word and two descriptions that do **not** explicitly contain either target word. Examples are intentionally compact and exploratory; matched lexical frequency, true semantic distance and generalization across models are **not guaranteed**.

In [ ]:
ITEMS = [
 {'x':'banana','y':'apple','related':'Monkeys are pictured holding a curved yellow fruit.',
  'x_desc':'a curved yellow fruit','y_desc':'a crisp round red fruit'},
 {'x':'elephant','y':'giraffe','related':'The huge gray animal has a trunk and big ears.',
  'x_desc':'a huge gray animal with a trunk','y_desc':'a tall spotted animal with a long neck'},
 {'x':'tiger','y':'lion','related':'A striped orange big cat prowls in the jungle.',
  'x_desc':'a striped orange big cat','y_desc':'a large cat with a thick mane'},
 {'x':'piano','y':'violin','related':'A classical keyboard instrument has black and white keys.',
  'x_desc':'a keyboard instrument played with both hands','y_desc':'a bowed string instrument held under the chin'},
 {'x':'coffee','y':'tea','related':'A dark drink is brewed from roasted beans.',
  'x_desc':'a drink brewed from roasted beans','y_desc':'a drink steeped from dried leaves'},
 {'x':'ocean','y':'river','related':'A vast body of salt water separates continents.',
  'x_desc':'a vast body of salt water','y_desc':'a freshwater current flowing toward the sea'},
 {'x':'castle','y':'palace','related':'A fortified medieval stone building has towers.',
  'x_desc':'a fortified medieval stone building','y_desc':'a grand royal residence with decorated halls'},
 {'x':'bicycle','y':'scooter','related':'A two-wheeled vehicle moves when the pedals turn.',
  'x_desc':'a two-wheeled vehicle with pedals','y_desc':'a small vehicle with a footboard and handlebars'},
]
UNRELATED = 'The train arrived at the station. Passengers checked their tickets.'
CONDITIONS = ['neutral','affirm','suppression','replacement']
PLURALS = dict(banana='bananas',elephant='elephants',tiger='tigers',piano='pianos',
               coffee='coffees',ocean='oceans',castle='castles',bicycle='bicycles')

def mention_target(response,x):
    forms = [x, PLURALS[x]]
    return bool(re.search(r'(?<![A-Za-z])(?:'+ '|'.join(map(re.escape,forms)) +r')(?![A-Za-z])',response,flags=re.I))

def instruction(x,y,c):
    return {'neutral':f'The topic word is {x}.',
            'affirm':f'Think of {x}.',
            'suppression':f'Do not think of {x}. Do not write the word {x}.',
            'replacement':f'Think of {y} instead of {x}. Do not write the word {x}.'}[c]

def mann_prompt(item,cond,cue,level):
    assert (cue == 'none') == (level == 0)
    text= '' if level==0 else ((item['related'] if cue=='related' else UNRELATED)+'\n')*level
    return instruction(item['x'],item['y'],cond)+'\n'+text+'Write one short sentence about whatever comes to mind.'

# Essential regression tests (the v2 code missed plural forms and counted zero twice).
assert mention_target('I bought bananas yesterday.','banana')
assert mention_target('The ELEPHANTS gathered.','elephant')
assert not mention_target('bananagrams are fun','banana')
assert mann_prompt(ITEMS[0],'suppression','none',0) == mann_prompt(ITEMS[0],'suppression','none',0)
assert all(item['x'] not in item['x_desc'] and item['y'] not in item['x_desc'] for item in ITEMS)
print('Stimulus sanity checks passed:',len(ITEMS),'targets')


## 2 — Mann-style free-generation experiment (behavior)

The **same random seed** is used across conditions for the same concept / distractor / replicate. This does not make generated responses identical, but reduces arbitrary seed assignment differences. Level 0 is sampled only once per condition; it is the common baseline for both distractor types.

**Primary measure:** whether the **singular or regular plural** of the forbidden target occurs in generated text. This is not a semantic mention detector (e.g., `the fruit` does not count). Show per-target results, not just the pooled mean.

In [ ]:
mann_rows=[]
for ci,item in enumerate(tqdm(ITEMS,desc='Mann-style generation')):
    for level in DISTRACTOR_LEVELS:
        cue_types=['none'] if level==0 else ['related','unrelated']
        for cue_type in cue_types:
            for rep in range(N_GENERATIONS):
                # Fixed across instruction conditions, not based on condition ID.
                base_seed=SEED+ci*100000+level*1000+rep+ (0 if cue_type in ['none','related'] else 100)
                for cond in CONDITIONS:
                    text=mann_prompt(item,cond,cue_type,level)
                    answer=generate_one(text, base_seed)
                    mann_rows.append(dict(concept=item['x'],replacement=item['y'],condition=cond,
                        distractor=cue_type,blocks=level,replicate=rep,seed=base_seed,
                        response=answer,target_mentioned=int(mention_target(answer,item['x'])),prompt=text))
mann_df=pd.DataFrame(mann_rows)
mann_df.to_csv(OUTPUT_DIR/'mann_generations_v3.csv',index=False)
mann_group=mann_df.groupby(['distractor','blocks','condition'],as_index=False).target_mentioned.agg(['mean','sum','count'])
display(mann_group)

fig,axs=plt.subplots(1,2,figsize=(12,4),sharey=True)
for ax,cue_type in zip(axs,['related','unrelated']):
    for cond in CONDITIONS:
        subset=mann_group[(mann_group.condition==cond) & ((mann_group.distractor==cue_type) | (mann_group.distractor=='none'))]
        subset=subset.sort_values('blocks')
        ax.plot(subset.blocks,subset['mean']*100,marker='o',label=cond)
    ax.set(xlabel='Distractor blocks',ylabel='Target mentioned (%)',title=f'{cue_type} distractor',ylim=(0,100))
axs[0].legend();write_plot('mann_mention_rates_v3.png')

# Per-concept frequencies matter; 5 stochastic draws per item are NOT five independent concepts.
perconcept=mann_df.groupby(['concept','condition','distractor','blocks'],as_index=False).target_mentioned.mean()
perconcept.to_csv(OUTPUT_DIR/'mann_per_concept_v3.csv',index=False)


## 3 — Associative-cue response likelihood (behavior)

For each concept, score the same candidate word under **the same question** in each instruction condition. Compare `suppression − neutral` and `replacement − suppression`. These log probabilities are **surface form likelihoods**, not neuron activations. The candidate word is kept fixed within each paired comparison.

In [ ]:
QUESTIONS = {
 'banana':('Which yellow fruit is associated with cartoon monkeys?','Which instrument has black and white keys?'),
 'elephant':('Which huge animal has a trunk?','Which drink comes from roasted beans?'),
 'tiger':('Which orange striped big cat lives in Asia?','What two-wheeled vehicle has pedals?'),
 'piano':('Which keyboard instrument is played in classical music?','Which animal has a trunk?'),
 'coffee':('Which drink is brewed from roasted beans?','Which medieval building has towers?'),
 'ocean':('Which vast saltwater body separates continents?','Which instrument has pedals?'),
 'castle':('Which medieval fortress has towers?','Which curved fruit is yellow?'),
 'bicycle':('Which two-wheeled vehicle has pedals?','Which vast body of saltwater separates continents?')}

cue_rows=[]
for item in tqdm(ITEMS,desc='Associative cue likelihood'):
    for cond in CONDITIONS:
        for cue,q in zip(['related','unrelated'],QUESTIONS[item['x']]):
            user=instruction(item['x'],item['y'],cond)+'\n'+q+'\nAnswer with exactly one word.'
            lp,n=continuation_logprob(user,item['x'])
            cue_rows.append(dict(concept=item['x'],condition=cond,cue=cue,logprob=lp,
                                 num_tokens=n,prompt=user))
cue_df=pd.DataFrame(cue_rows);cue_df.to_csv(OUTPUT_DIR/'cue_logprob_v3.csv',index=False)
wide=cue_df.pivot(index=['concept','cue'],columns='condition',values='logprob').reset_index()
wide['suppression_minus_neutral']=wide.suppression-wide.neutral
wide['replacement_minus_suppression']=wide.replacement-wide.suppression
wide.to_csv(OUTPUT_DIR/'cue_paired_contrasts_v3.csv',index=False)
display(wide.groupby('cue')[['suppression_minus_neutral','replacement_minus_suppression']].agg(['mean','std','count']).round(3))
fig,ax=plt.subplots(figsize=(9,4))
for cue in ['related','unrelated']:
    d=wide[wide.cue==cue]
    cols=['suppression_minus_neutral','replacement_minus_suppression']
    ax.errorbar(cols,d[cols].mean().values,yerr=d[cols].sem().values,capsize=3,marker='o',label=cue)
ax.axhline(0,color='gray',linestyle='--');ax.set(title='Within-concept paired log-probability differences',ylabel='Delta log P(target)')
ax.legend();write_plot('cue_direct_comparison_v3.png')


## 4 — Zhou-style negation: words versus answer-label bias

Six groups of 3 words, 3 rotations of the same option list, and 3 correct-answer positions. With both polarities this gives **108 paired conditions**, but only **6 distinct word groups**, so do not treat all 108 as independent.

**Primary behavioral test:** compare the conditional likelihood of **full answer words**. This largely removes the A/B/C label prior, though tokenization and lexical frequency biases remain. Record token counts and exclude / stratify multi-token items when comparing candidates. **Diagnostics:** raw A/B/C scores and a context-calibrated (baseline-subtracted) label score. A change in accuracy after calibration is not proof of negation understanding.


In [ ]:
WORD_SETS=[['gas','liquid','solid'],['red','blue','green'],['metal','wood','plastic'],
           ['banana','apple','orange'],['cat','dog','bird'],['summer','winter','spring']]
LABELS=['A','B','C']

def zhou_user(opts,gold,polarity,response_mode='word',baseline=False):
    others=[word for i,word in enumerate(opts) if i!=gold]
    statement=('The correct option is '+opts[gold]+'.' if polarity=='positive'
               else 'The correct option is not '+others[0]+' and not '+others[1]+'. Exactly one option is correct.')
    if baseline:statement='Read the options carefully.'
    options='\n'.join(f'{label}) {word}' for label,word in zip(LABELS,opts))
    suffix=('Reply with ONLY the exact option WORD, not the letter.' if response_mode=='word'
            else 'Reply with ONLY one letter: A, B or C.')
    return 'Choose based on the statement.\n'+statement+'\n'+options+'\n'+suffix

@torch.inference_mode()
def label_scores(prompt):
    pref=chat(prompt); prefix=tok(pref,add_special_tokens=False).input_ids
    ids=[]
    for label in LABELS:
        full=tok(pref+label,add_special_tokens=False).input_ids
        if full[:len(prefix)]!=prefix or len(full)!=len(prefix)+1:
            raise ValueError('Option label tokenization is not a single-token continuation: '+label)
        ids.append(full[-1])
    logits=model(torch.tensor([prefix],device=DEVICE)).logits[0,-1,ids].float().cpu().numpy()
    return logits, ids

def softmax_np(logits):
    logits=np.asarray(logits,dtype=float)
    w=np.exp(logits-np.max(logits));return w/w.sum()

zrows=[]
word_baseline_cache={}
for k,words in enumerate(tqdm(WORD_SETS,desc='Zhou-style negation')):
    for rotation in range(3):
        opts=words[rotation:]+words[:rotation]
        for gold in range(3):
            for polarity in ['positive','negative']:
                wp=zhou_user(opts,gold,polarity,'word')
                lp_counts=[continuation_logprob(wp,word) for word in opts]
                lp=np.array([e[0] for e in lp_counts])
                token_counts=[e[1] for e in lp_counts]
                # A/B/C mode, and context-calibrated A/B/C mode for bias diagnostics.
                ap=zhou_user(opts,gold,polarity,'letter')
                bp=zhou_user(opts,gold,polarity,'letter',baseline=True)
                letter_logits,_=label_scores(ap)
                base_logits,_=label_scores(bp)
                raw_p=softmax_np(letter_logits)
                calibrated_p=softmax_np(letter_logits-base_logits)
                word_p=softmax_np(lp)
                cache_key=(k,rotation)
                if cache_key not in word_baseline_cache:
                    null_word_prompt=zhou_user(opts,gold,polarity,'word',baseline=True)
                    word_baseline_cache[cache_key]=np.array([continuation_logprob(null_word_prompt,w)[0] for w in opts])
                word_adjusted=softmax_np(lp-word_baseline_cache[cache_key])
                # Contextual calibration is diagnostic, not a guarantee of unbiased inference.
                # Compare only words of identical token length in strict single-token subset analysis.
                zrows.append(dict(set_id=k,rotation=rotation,gold=gold,polarity=polarity,
                    correct_word=opts[gold],options='|'.join(opts),
                    word_prediction=opts[int(word_p.argmax())],word_correct=int(word_p.argmax()==gold),
                    word_calibrated_prediction=opts[int(word_adjusted.argmax())],
                    word_calibrated_correct=int(word_adjusted.argmax()==gold),
                    word_p_gold=float(word_p[gold]),word_token_counts='|'.join(map(str,token_counts)),
                    all_one_token=int(all(n==1 for n in token_counts)),
                    letter_prediction=LABELS[int(raw_p.argmax())],letter_correct=int(raw_p.argmax()==gold),
                    calibrated_prediction=LABELS[int(calibrated_p.argmax())],calibrated_correct=int(calibrated_p.argmax()==gold),
                    label_p_A=float(raw_p[0]),label_p_B=float(raw_p[1]),label_p_C=float(raw_p[2]),
                    calibrated_p_A=float(calibrated_p[0]),calibrated_p_B=float(calibrated_p[1]),calibrated_p_C=float(calibrated_p[2]),
                    prompt_word=wp,prompt_letter=ap))
z_df=pd.DataFrame(zrows);z_df.to_csv(OUTPUT_DIR/'zhou_behavior_v3.csv',index=False)
print('Accuracy (all items):')
display(z_df.groupby('polarity')[['word_correct','word_calibrated_correct','letter_correct','calibrated_correct','word_p_gold']].agg(['mean','count']).round(3))
print('Single-token-only subset count:',int(z_df.all_one_token.sum()),'of',len(z_df))
if z_df.all_one_token.any():
    display(z_df[z_df.all_one_token==1].groupby('polarity')[['word_correct','word_calibrated_correct','letter_correct','calibrated_correct']].mean().round(3))
print('Confusion by correct position — raw label mode:')
display(pd.crosstab([z_df.polarity,z_df.gold],z_df.letter_prediction))
print('Confusion by correct position — calibrated mode:')
display(pd.crosstab([z_df.polarity,z_df.gold],z_df.calibrated_prediction))

fig,axs=plt.subplots(1,2,figsize=(12,4),sharey=True)
for ax,pol in zip(axs,['positive','negative']):
    vals=z_df[z_df.polarity==pol][['word_correct','word_calibrated_correct','letter_correct','calibrated_correct']].mean()
    ax.bar(['Words','Words cal.','Letters raw','Letters cal.'],vals.values)
    ax.set(ylim=(0,1),ylabel='Accuracy',title=pol)
    ax.tick_params(axis='x',labelrotation=15)
write_plot('zhou_accuracy_methods_v3.png')


## 5 — Zhou-style layer-wise **label** Logit Lens (diagnostic)

This reproduces the *method of looking at hidden states* rather than the authors' causal mechanism experiment. Do not interpret A/B/C predictions as semantic concept activation. The last layer is taken from `result.logits` to avoid **double final normalization**, the bug fixed in v2.

In [ ]:
@torch.inference_mode()
def zhou_lens(prompt):
    normal_logits,ids=label_scores(prompt)
    enc=tok(chat(prompt),return_tensors='pt',add_special_tokens=False).to(DEVICE)
    result=model(**enc,output_hidden_states=True,return_dict=True)
    hidden=result.hidden_states
    values=[]
    for i,h in enumerate(hidden):
        if i==len(hidden)-1:
            selected=result.logits[0,-1,ids].float()
        else:
            selected=model.lm_head(model.model.norm(h[:,-1,:]))[0,ids].float()
        values.append(selected.softmax(-1).cpu().numpy())
    values=np.stack(values)
    np.testing.assert_allclose(values[-1],softmax_np(normal_logits),atol=1e-3,rtol=1e-3)
    return values

lens_rows=[]
for k,words in enumerate(tqdm(WORD_SETS,desc='Zhou layer lens')):
    for polarity in ['positive','negative']:
        # Keep the target at A for the illustrative layer comparison; this is not a full positional control.
        arr=zhou_lens(zhou_user(words,0,polarity,'letter'))
        for layer,support in enumerate(arr):
            lens_rows.append(dict(set_id=k,polarity=polarity,layer=layer,
                                  p_A=float(support[0]),p_B=float(support[1]),p_C=float(support[2])))
lens_df=pd.DataFrame(lens_rows);lens_df.to_csv(OUTPUT_DIR/'zhou_layer_lens_v3.csv',index=False)
fig,axes=plt.subplots(1,2,figsize=(12,4),sharey=True)
for ax,pol in zip(axes,['positive','negative']):
    d=lens_df[lens_df.polarity==pol].groupby('layer')[['p_A','p_B','p_C']].mean()
    for label in LABELS:ax.plot(d.index,d['p_'+label],label=label)
    ax.set(title=pol,xlabel='Hidden state index (0 = embedding)',ylim=(0,1))
    ax.legend()
axes[0].set_ylabel('Mean answer-label support')
write_plot('zhou_layer_lens_v3.png')
print('Saved',len(lens_df),'layer entries')


## 6 — Concept-level linear probes (new, exploratory)

**What is trained?** For each fixed (X,Y) pair, use a small regularized logistic classifier to distinguish residual-stream representations elicited by **X** vs **Y** in varied sentence templates, at the same terminal marker `Topic:`. The probing task is **paired X-vs-Y discrimination**, not a universal binary detector for X.

**Validation:** Evaluate on (1) held-out text templates naming X/Y, and separately (2) *descriptions that do not contain either name*. If performance is poor, **do not** claim that the probe measures a semantic concept. High performance still does not prove a causal mechanism or memory loss.

**Study variable:** On suppression/affirm/neutral/replacement prompts, measure how the **probe decision score** for X-vs-Y changes with related vs unrelated context. This may include literal word traces from the instruction; always compare within-concept controls and report limitations.


In [ ]:
TRAIN_FRAMES=[
    'The note concerns {}.\nTopic:',
    'Think of {}.\nTopic:',
    'An example topic is {}.\nTopic:',
    'Someone mentioned {} earlier.\nTopic:',
    'The main subject is {}.\nTopic:',
    'The concept to discuss is {}.\nTopic:',
    'Keep {} in mind.\nTopic:',
    'Focus attention on {}.\nTopic:',
]
HOLDOUT_FRAMES=[
    'A writer spoke about {}.\nTopic:',
    'The conversation revolves around {}.\nTopic:',
    'The word of interest is {}.\nTopic:',
]
DESCRIPTION_FRAMES=[
    'A writer described {}.\nTopic:',
    'The main topic can be described as {}.\nTopic:',
    'Consider the idea of {}.\nTopic:',
]

@torch.inference_mode()
def feature_at_layers(user_text,layer_ids=PROBE_LAYERS):
    enc=tok(chat(user_text),return_tensors='pt',add_special_tokens=False).to(DEVICE)
    out=model(**enc,output_hidden_states=True,return_dict=True)
    # Last-position hidden states; final state is already normalized by model.
    return {layer:out.hidden_states[layer][0,-1,:].float().cpu().numpy() for layer in layer_ids}

# Build all datasets once; avoid model forward passes for every independent layer.
probe_examples=[]
for item in tqdm(ITEMS,desc='Probe train/held-out contexts'):
    for split,frames in [('train',TRAIN_FRAMES),('holdout_named',HOLDOUT_FRAMES),('holdout_description',DESCRIPTION_FRAMES)]:
        for frame in frames:
            for cls in [1,0]:
                target=item['x'] if cls==1 else item['y']
                entry=target if split!='holdout_description' else (item['x_desc'] if cls==1 else item['y_desc'])
                user=frame.format(entry)
                feats=feature_at_layers(user)
                probe_examples.append(dict(concept=item['x'],replacement=item['y'],split=split,y=cls,prompt=user,features=feats))
print('Probe examples:',len(probe_examples))

trained={}
probe_metrics=[]
for item in ITEMS:
    x=item['x'];trained[x]={}
    relevant=[z for z in probe_examples if z['concept']==x]
    for layer in PROBE_LAYERS:
        train=[z for z in relevant if z['split']=='train']
        X=np.stack([z['features'][layer] for z in train]); y=np.array([z['y'] for z in train])
        clf=make_pipeline(StandardScaler(),LogisticRegression(C=0.1,max_iter=2000,solver='liblinear',random_state=SEED))
        clf.fit(X,y)
        trained[x][layer]=clf
        for split in ['holdout_named','holdout_description']:
            ev=[z for z in relevant if z['split']==split]
            Z=np.stack([z['features'][layer] for z in ev]); gy=np.array([z['y'] for z in ev])
            scores=clf.decision_function(Z)
            probe_metrics.append(dict(concept=x,layer=layer,validation=split,n=len(ev),
                                      auc=float(roc_auc_score(gy,scores)),
                                      accuracy=float(accuracy_score(gy,scores>=0))))
probe_metric_df=pd.DataFrame(probe_metrics);probe_metric_df.to_csv(OUTPUT_DIR/'concept_probe_validation_v3.csv',index=False)
display(probe_metric_df.groupby(['layer','validation'])[['auc','accuracy']].agg(['mean','std']).round(3))
print('Description validation by concept:')
display(probe_metric_df[probe_metric_df.validation=='holdout_description'].pivot(index='concept',columns='layer',values='auc').round(2))


### 6B — Apply only validated probes to suppression and cue conditions

The score represents a **model-dependent X-vs-Y log-odds**, not the probability a concept is consciously recalled. Results are shown for **all** probes with an explicit reliability flag (`AUC ≥ 0.75` on name-free descriptions, a heuristic screen chosen in advance); the unfiltered results are also saved to avoid selective reporting.

Primary exploratory contrast: `[(suppression related − suppression unrelated) − (neutral related − neutral unrelated)]`. This difference-in-differences controls the response to cues in the absence of explicit suppression, although it does not remove all effects of prompt length or instruction wording.

In [ ]:
PROBE_AUC_THRESHOLD=0.75
measurement=[]
for item in tqdm(ITEMS,desc='Measure concept probe scores'):
    x,y=item['x'],item['y']
    for cond in CONDITIONS:
        for cue in ['none','related','unrelated']:
            distractor='' if cue=='none' else (item['related'] if cue=='related' else UNRELATED)
            # Fixed ending marker in all train, validation, and test contexts.
            p=instruction(x,y,cond)+'\n'+(distractor+'\n' if distractor else '')+'Topic:'
            feats=feature_at_layers(p)
            for layer in PROBE_LAYERS:
                clf=trained[x][layer]
                score=float(clf.decision_function(feats[layer].reshape(1,-1))[0])
                auc=float(probe_metric_df.query('concept==@x and layer==@layer and validation=="holdout_description"').auc.iloc[0])
                measurement.append(dict(concept=x,replacement=y,condition=cond,cue=cue,layer=layer,
                                        probe_log_odds=score,validation_description_auc=auc,
                                        validated=int(auc>=PROBE_AUC_THRESHOLD),prompt=p))
measure_df=pd.DataFrame(measurement);measure_df.to_csv(OUTPUT_DIR/'concept_probe_measurements_v3.csv',index=False)

pv=measure_df.pivot(index=['concept','layer','cue','validated'],columns='condition',values='probe_log_odds').reset_index()
pv['suppression_minus_neutral']=pv.suppression-pv.neutral
pv['replacement_minus_suppression']=pv.replacement-pv.suppression
pv.to_csv(OUTPUT_DIR/'concept_probe_paired_contrasts_v3.csv',index=False)
# Difference in differences on identical cues, per concept and layer.
did=pv.pivot(index=['concept','layer','validated'],columns='cue',values='suppression_minus_neutral').reset_index()
did['related_minus_unrelated_DID']=did['related']-did['unrelated']
did.to_csv(OUTPUT_DIR/'concept_probe_DID_v3.csv',index=False)
print('Valid concept/layer probes:',int(probe_metric_df.query('validation=="holdout_description"').auc.ge(PROBE_AUC_THRESHOLD).sum()),
      '/',len(ITEMS)*len(PROBE_LAYERS))
print('All probes — descriptive only:')
display(did.groupby('layer')['related_minus_unrelated_DID'].agg(['mean','std','count']).round(3))
if did.validated.any():
    print('AUC-filtered exploratory summary (do not discard unfiltered CSV):')
    display(did[did.validated==1].groupby('layer')['related_minus_unrelated_DID'].agg(['mean','std','count']).round(3))
else:
    print('No probes passed the prespecified semantic validation threshold; do NOT interpret score changes as semantic concept activation.')

fig,ax=plt.subplots(figsize=(8,4))
for layer in PROBE_LAYERS:
    s=did[did.layer==layer]
    ax.scatter([layer]*len(s),s.related_minus_unrelated_DID,s=28,alpha=0.65)
    ax.plot(layer,s.related_minus_unrelated_DID.mean(),'kx',markersize=9)
ax.axhline(0,c='gray',ls='--');ax.set(xlabel='Hidden-state index',ylabel='Suppression vs neutral: related − unrelated',
 title='Concept-probe difference-in-differences (all concepts; exploratory)')
write_plot('concept_probe_DID_v3.png')


## 7 — Paired uncertainty intervals (exploratory)

The independent resampling unit is **the concept (Mann, n=8)** or **the word set (Zhou, n=6)**, not the individual randomly sampled sentence or rotated multiple-choice item. The bootstrap intervals are descriptive with very small numbers of independent items; they are **not** preregistered hypothesis tests.

- Mann: suppression-minus-replacement forbidden-mention rate for **related cue, one distractor block**, average within each concept across samples.
- Zhou: positive-minus-negative **word-answer accuracy**, averaged within each original word set before resampling. Includes uncalibrated and calibrated word-choice diagnostics.


In [ ]:
def cluster_bootstrap(contrasts, reps=5000, seed=SEED):
    values=np.asarray(contrasts,dtype=float)
    rng=np.random.default_rng(seed)
    if len(values)<2: return float(values.mean()),float('nan'),float('nan')
    draws=rng.choice(values,size=(reps,len(values)),replace=True).mean(axis=1)
    return float(values.mean()),float(np.quantile(draws,.025)),float(np.quantile(draws,.975))

# concept-level paired rates: evaluate only the genuinely manipulated related-cue level 1.
sub=mann_df[(mann_df.blocks==1)&(mann_df.distractor=='related')]
mrate=sub.groupby(['concept','condition']).target_mentioned.mean().unstack()
mann_diff=mrate.suppression-mrate.replacement
# group by each original set; no pseudoreplication across different label rotations.
zg=z_df.groupby(['set_id','polarity'])[['word_correct','word_calibrated_correct']].mean().unstack('polarity')
z_word=zg[('word_correct','positive')]-zg[('word_correct','negative')]
z_word_cal=zg[('word_calibrated_correct','positive')]-zg[('word_calibrated_correct','negative')]

pairs=[('mann_suppression_minus_replacement_related_level1',mann_diff),
       ('zhou_positive_minus_negative_word_accuracy',z_word),
       ('zhou_positive_minus_negative_calibrated_word_accuracy',z_word_cal)]
boot=[]
for name,series in pairs:
    mean,lower,upper=cluster_bootstrap(series.values)
    boot.append(dict(contrast=name,n_clusters=int(series.shape[0]),estimate=mean,ci95_low=lower,ci95_high=upper))
bootstrap_df=pd.DataFrame(boot)
bootstrap_df.to_csv(OUTPUT_DIR/'cluster_bootstrap_v3.csv',index=False)
display(bootstrap_df.round(3))
print('Warning: these intervals are exploratory and may be unstable with only 6–8 independent clusters.')


## 7 — Summary, full artifact package, and reproducibility checks

Raw data is included even when an experiment shows null, noisy or unexpected results. Dataset counts should match the fixed design. The ZIP contains CSVs, figures, metadata and a human-readable summary. This notebook does not embed model weights or private credentials.

In [ ]:
# Counts and data-quality assertions
expected_mann=len(ITEMS)*len(CONDITIONS)*N_GENERATIONS*(1+2*(len(DISTRACTOR_LEVELS)-1))
assert len(mann_df)==expected_mann,(len(mann_df),expected_mann)
assert len(cue_df)==len(ITEMS)*len(CONDITIONS)*2
assert len(z_df)==len(WORD_SETS)*3*3*2
assert len(lens_df)==len(WORD_SETS)*2*(model.config.num_hidden_layers+1)
assert len(probe_metric_df)==len(ITEMS)*len(PROBE_LAYERS)*2
assert len(measure_df)==len(ITEMS)*len(CONDITIONS)*3*len(PROBE_LAYERS)

summary={
 'mann_total_generations':int(len(mann_df)),
 'cluster_bootstrap':bootstrap_df.to_dict(orient='records'),
 'mann_rates':mann_group.to_dict(orient='records'),
 'zhou_accuracy':z_df.groupby('polarity')[['word_correct','word_calibrated_correct','letter_correct','calibrated_correct']].mean().round(5).to_dict('index'),
 'zhou_all_one_token_fraction':float(z_df.all_one_token.mean()),
 'probe_semantic_validation_auc':probe_metric_df.query('validation=="holdout_description"').groupby('layer').auc.mean().to_dict(),
 'probes_passing_auc_threshold':int(probe_metric_df.query('validation=="holdout_description"').auc.ge(PROBE_AUC_THRESHOLD).sum()),
 'concept_probe_DID_all':did.groupby('layer').related_minus_unrelated_DID.mean().to_dict(),
}
metadata={'title':'Mann / Zhou / concept-probe exploratory pilot v3',
 'model':MODEL_ID,'seed':SEED,'N_GENERATIONS':N_GENERATIONS,'distractor_levels':DISTRACTOR_LEVELS,
 'probe_layers':PROBE_LAYERS,'probe_auc_threshold':PROBE_AUC_THRESHOLD,
 'python':sys.version,'torch':torch.__version__,'transformers':__import__('transformers').__version__,
 'numpy':np.__version__,'device':DEVICE,'run_completed_utc':datetime.now(timezone.utc).isoformat(),
 'sources':['https://arxiv.org/abs/2511.12381','https://proceedings.mlr.press/v306/zhou26bq.html'],
 'disclaimer':'Exploratory analogue only; not original paper replication or causal mechanism identification.'}
(OUTPUT_DIR/'metadata.json').write_text(json.dumps(metadata,ensure_ascii=False,indent=2),encoding='utf-8')
(OUTPUT_DIR/'summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')

text_lines=['# White Bear / Zhou v3 — exploratory run','',
 f'Model: `{MODEL_ID}`; seed: `{SEED}`; generation samples: `{N_GENERATIONS}` per unique prompt.',
 '', '## Behavioral performance (Zhou-inspired)','',
 '| Polarity | Word answers | Word answers (calibrated) | Letter answers (raw) | Letter answers (calibrated) |',
 '|---|---:|---:|---:|---:|']
for polarity,row in summary['zhou_accuracy'].items():
    text_lines.append(f"| {polarity} | {row['word_correct']:.1%} | {row['word_calibrated_correct']:.1%} | {row['letter_correct']:.1%} | {row['calibrated_correct']:.1%} |")
text_lines += ['', '## Interpretive limits',
 '- Mann: forbidden words counted as singular or explicitly listed plural forms; semantic paraphrases not counted.',
 '- Zhou: answer-word scores can still reflect lexical and token-length biases; only 6 word sets.',
 '- Concept probe: X-versus-Y discrimination; use semantic-description held-out AUC before interpreting.',
 '- This run is NOT an exact replication of either paper. No causal suppression/construction mechanism is identified.',
 '- Exploratory paired uncertainty intervals resample independent concepts / word sets, not repeated draws.',
 '- Every CSV and graph is preserved, including unexpected outcomes.', '']
(OUTPUT_DIR/'README.md').write_text('\n'.join(text_lines),encoding='utf-8')

ZIP_NAME='white_bear_zhou_pilot_v3_results.zip'
with zipfile.ZipFile(ZIP_NAME,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for path in sorted(OUTPUT_DIR.rglob('*')):
        if path.is_file():z.write(path,arcname=str(path))
print('Output ZIP:', ZIP_NAME, 'size KB:', round(os.path.getsize(ZIP_NAME)/1024,1))
print('Generated CSV files:', len(list(OUTPUT_DIR.glob('*.csv'))), 'PNG files:',len(list(OUTPUT_DIR.glob('*.png'))))
print((OUTPUT_DIR/'README.md').read_text())


## 8 — GitHub upload (secure; enabled)

You can save results to **`ryouy/GraduationThesis/1015/results_v3/<UTC timestamp>/`** directly from Colab. The folder contains the ZIP plus README/metadata. All CSV and PNG files are *inside* the ZIP (avoid dozens of separate GitHub commits).

**One-time setup (Colab):**
1. Create a **fine-grained Personal Access Token** in GitHub → Settings → Developer settings → Personal access tokens → Fine-grained tokens. Restrict it to **`ryouy/GraduationThesis`** and grant **Repository permissions → Contents: Read and write**. Prefer an expiration date.
2. In Colab, open the **Secrets (key icon)** panel. Add `GITHUB_RESULTS_TOKEN` with the token value and **enable notebook access**. Never paste the token into a code cell, prompt or GitHub commit.
3. GitHub upload is enabled; execute the next cell after experiments. The repo is public: do **not** upload private content.

Only this final cell needs GitHub authentication. If secrets are unavailable or upload is disabled, the ZIP remains local to Colab and you can use `files.download(...)`.


In [ ]:
UPLOAD_TO_GITHUB = True          # Explicit opt-in; set True only after configuring Colab Secrets
REPOSITORY = 'ryouy/GraduationThesis'
BRANCH = 'main'
GITHUB_SECRET_NAME = 'GITHUB_RESULTS_TOKEN'

def github_upload_file(token, repo, path, data, branch='main'):
    import base64, requests
    endpoint=f'https://api.github.com/repos/{repo}/contents/{path}'
    headers={'Authorization':'Bearer '+token,
             'Accept':'application/vnd.github+json',
             'X-GitHub-Api-Version':'2022-11-28'}
    payload={'message':f'Archive v3 thesis pilot results: {path}',
             'content':base64.b64encode(data).decode('ascii'),'branch':branch}
    # Create new dated directory; never silently overwrite existing experimental data.
    response=requests.put(endpoint,headers=headers,json=payload,timeout=90)
    if response.status_code not in (200,201):
        raise RuntimeError(f'GitHub upload failed (HTTP {response.status_code}): {response.text[:500]}')
    return response.json()['content']['html_url']

if UPLOAD_TO_GITHUB:
    try:
        from google.colab import userdata
        token=userdata.get(GITHUB_SECRET_NAME)
    except Exception as e:
        raise RuntimeError('Open Colab Secrets, add GITHUB_RESULTS_TOKEN and allow notebook access.') from e
    if not token or not token.strip():
        raise RuntimeError('GITHUB_RESULTS_TOKEN is empty.')
    stamp=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    prefix=f'1015/results_v3/{stamp}'
    files_to_push=[Path(ZIP_NAME),OUTPUT_DIR/'README.md',OUTPUT_DIR/'metadata.json']
    urls=[]
    for file_path in files_to_push:
        url=github_upload_file(token,REPOSITORY,f'{prefix}/{file_path.name}',file_path.read_bytes(),branch=BRANCH)
        urls.append(url)
    print('Upload complete. Files:')
    for url in urls:print(url)
    del token
else:
    print('GitHub upload is OFF. Results are saved as:',ZIP_NAME)
    print('To download manually in Colab, run:')
    print("from google.colab import files; files.download('"+ZIP_NAME+"')")
